# v129 — `320×320` Spine-Guided EdgeAttUNet Refiner 厳密対照実験 (Idea 5)

**Colab L4 約 15〜16 分で完結**（`v127` / `v130` と完全分離された保存先で同時並列実行可能）。

### なぜ `v129` が有望か（`v128` 失敗からの構造的フィードバック）
- `v128` により、推論時に `proposal` の外側へ踏み出す後処理（Hysteresis）や PCA 直線分割は隣接フィラメント混入・湾曲単一チャネル誤分割を招き、`shrink_mask` (`(prob >= 0.45) & proposal`) が推論契約として最適であることが確定した。
- 一方、`v127` は YOLO の粗い `160×160` P3 グリッド（`12.8 px`/cell）にのみ `spine` 補助教師を入れており、最終マスク境界を決定する **`320×320` 高解像度クロップ Refiner（`1〜2 px`/cell）には一度も `spine` 教師が使われていない**。
- 本実験では **推論アーキテクチャと `shrink_mask` (`thr=0.45`) を 100% 無変更に保ったまま**、同一初期重み・同一クロップ順序で以下 3 系統の `MediSegEdgeAttUNet` (各 14 epoch, 約 4 分) を厳密対照学習する:
  1. `ref_ctrl`: `aux_weight=0.0, spine_gamma=0.0`（現行 `prod` と完全同一損失の対照群）
  2. `ref_spine_aux`: `aux_weight=0.25, spine_gamma=0.0`（`up4` `320×320` 特徴への `1×1 spine_head` マルチタスク表現学習のみ）
  3. `ref_spine_bal`: `aux_weight=0.25, spine_gamma=1.5`（`spine_head` + 前景質量保存型の中心線強調 BCE `sum_{x in M} w(x) == |M|`）


In [ ]:
# [1/6] 依存パッケージ・Drive マウント・実験設定 (v127 / v130 と完全分離)
%pip install -q "ultralytics==8.4.120" pycocotools scipy
import hashlib, io, json, math, os, pickle, random, shutil, sys, time
from collections import defaultdict
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from pycocotools import mask as mask_utils
from tqdm.auto import tqdm
from ultralytics import YOLO

drive.mount("/content/drive")

class CFG:
    SPLIT_MODE = "pilot"  # "pilot" (365 dev fit / 180 dev∩fold2 eval, locked_eval 非接触) or "temporal_confirm" (545 fit / 160 locked eval)
    SEED = 3407

    GROUPED_FOLDS_SHA256 = "febb3f67237a058fa8a71f188fac262035048b55d860c0e4509514877f6af67e"
    TIME_BLOCK_SHA256 = "9a9b17252133c9500d44bf3b6ec56879a47e4b32d45cb14be080ba29148c732d"
    EXCLUSIONS_SHA256 = "a56372ea6493a6d3571910e60f291606b864a4f84a61baf1df475a6627270b0a"

    DRIVE_ROOT = Path("/content/drive/MyDrive/solar_filament_v30")
    DATA_CACHE = DRIVE_ROOT / "competition_data"
    ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
    PREPROC_CACHE = DRIVE_ROOT / "preprocessed_v30_q96"
    EXP_DIR = DRIVE_ROOT / "runs" / "v129_spine_refiner"
    LOCAL_ROOT = Path("/content/solar_v129")
    PREPROC_DIR = LOCAL_ROOT / "preprocessed"
    PROPOSAL_CACHE_DIR = LOCAL_ROOT / "oof_proposals"

    CROP_SIZE = 320
    CROP_CONTEXT = 1.65
    SPINE_RADIUS_PX = 2
    REFINER_EPOCHS = 14
    REFINER_BATCH = 8
    REFINER_LR = 2e-4
    REFINER_WD = 1e-4

    YOLO_IMGSZ = 1280
    PRIMARY_CONF = 0.26
    PREDICT_IOU = 0.55
    PRIMARY_THR = 0.45
    DIAG_THRS = (0.42, 0.45, 0.48)
    DROP_THRESHOLD = 0.90
    MIN_AREA = 20
    SMALL_GT_MAX_AREA = 1024.0
    SPARSE_R = 0.77
    BOOTSTRAP_ITERS = 2000

    ARMS = {
        "ref_ctrl":      {"aux_weight": 0.00, "spine_gamma": 0.0},
        "ref_spine_aux": {"aux_weight": 0.25, "spine_gamma": 0.0},
        "ref_spine_bal": {"aux_weight": 0.25, "spine_gamma": 1.5},
    }

for p in (CFG.LOCAL_ROOT, CFG.PREPROC_DIR, CFG.PROPOSAL_CACHE_DIR, CFG.EXP_DIR):
    p.mkdir(parents=True, exist_ok=True)

random.seed(CFG.SEED); np.random.seed(CFG.SEED); torch.manual_seed(CFG.SEED)
assert torch.cuda.is_available(), "GPU ランタイムを選択してください"
RUNTIME_INFO = {
    "started_utc": datetime.now(timezone.utc).isoformat(),
    "gpu": torch.cuda.get_device_name(0),
    "split_mode": CFG.SPLIT_MODE,
    "python": sys.version,
    "torch": torch.__version__,
    "ultralytics": version("ultralytics"),
}
print(json.dumps(RUNTIME_INFO, indent=2, ensure_ascii=False))


In [ ]:
# [2/6] MediSegEdgeAttUNet + SpineGuidedEdgeAttUNet / 質量保存型 Spine Refiner 損失
"""
MediSeg-AI Knowledge Port for Solar Filament Segmentation
Includes:
1. MediSeg UNet
2. MediSeg DeepLabV3Plus with ASPP (Atrous Spatial Pyramid Pooling)
3. CombinedLoss (BCE + Dice Loss)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional

# ============================================================
# 1. Combined Loss (BCE + Dice Loss)
# ============================================================
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        
        intersection = (probs * targets).sum()
        dice = (2.0 * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1.0 - dice

class CombinedLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

# ============================================================
# 2. MediSeg UNet Architecture
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, mid_channels: Optional[int] = None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class MediSegUNet(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1, bilinear: bool = False):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

# ============================================================
# 3. MediSeg DeepLabV3Plus with ASPP Module
# ============================================================
class ASPP(nn.Module):
    def __init__(self, in_channels: int, out_channels: int = 256):
        super().__init__()
        self.conv1x1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1x1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=6, dilation=6, bias=False)
        self.bn3x3_1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_2 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=12, dilation=12, bias=False)
        self.bn3x3_2 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_3 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=18, dilation=18, bias=False)
        self.bn3x3_3 = nn.BatchNorm2d(out_channels)
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv_global = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn_global = nn.BatchNorm2d(out_channels)
        
        self.conv_final = nn.Conv2d(out_channels * 5, out_channels, kernel_size=1, bias=False)
        self.bn_final = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        size = x.shape[2:]
        x1 = self.relu(self.bn1x1(self.conv1x1(x)))
        x2 = self.relu(self.bn3x3_1(self.conv3x3_1(x)))
        x3 = self.relu(self.bn3x3_2(self.conv3x3_2(x)))
        x4 = self.relu(self.bn3x3_3(self.conv3x3_3(x)))
        
        x5 = self.global_avg_pool(x)
        x5 = self.relu(self.bn_global(self.conv_global(x5)))
        x5 = F.interpolate(x5, size=size, mode='bilinear', align_corners=True)
        
        x_cat = torch.cat([x1, x2, x3, x4, x5], dim=1)
        return self.relu(self.bn_final(self.conv_final(x_cat)))

class MediSegDeepLabV3Plus(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1):
        super().__init__()
        # Feature Extractor
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 512, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )
        self.aspp = ASPP(512, 256)
        self.classifier = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, out_channels, kernel_size=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_size = x.shape[2:]
        feat = self.stem(x)
        aspp_feat = self.aspp(feat)
        out = self.classifier(aspp_feat)
        return F.interpolate(out, size=input_size, mode='bilinear', align_corners=True)

# ============================================================
# 4. Edge-Guided Attention U-Net (EG-MHSA)
# ============================================================
# 2026-08-29、WINNING_STRATEGY_2026-08-29.md §1.4のPhase 2実験用に追加。
# 参考文献: "EdgeAttNet: Towards Barb-Aware Filament Segmentation"
# (arXiv:2509.02964)。MAGFiLO上でU-Net(mIoU_pairwise 0.5724)から
# EdgeAttNet(0.6451)への改善を報告しており、barb(フィラメントの微細な
# 枝構造)の認識精度向上を狙ったアーキテクチャ。本プロジェクトの
# refiner(MediSegUNet)が抱える主要な弱点(near-miss境界品質)と直接
# 対応するため、置き換え候補として追試する。
#
# 【重要な注意】著者の公開リポジトリ(github.com/dasjar/EdgeAttNet)を
# 直接確認したところ、`model/edgeattnet_model.py`(本来UNetEdgeTransformer
# クラスが定義されているべきファイル)の中身が誤って`main.py`と同一に
# なっており、実際のアーキテクチャコードはリポジトリ上に存在しない
# (2026-08-29、`gh api`でリポジトリ全ファイルを走査して確認)。学習済み
# 重みの配布も確認できなかった。**以下は論文本文のアーキテクチャ記述
# (edge prior E=σ(Conv3x3(入力画像))をbottleneckへbilinear補間し
# 1x1convで射影、Q・Kへ加算、位置エンコーディングなし)のみを根拠にした
# 独自の再実装であり、著者オリジナル実装とのコード上の一致は未検証。**
# ベースにするU-Net本体はMediSegUNetをそのまま流用しているため、
# パラメータ総数は論文記載の値(22.6M)とは一致しない。


class EdgeGuidedMHSA(nn.Module):
    """Edge-Guided Multi-Head Self-Attention (EG-MHSA)。

    U-NetのbottleneckでSelf-Attentionを行う際、入力画像から抽出した
    エッジ事前分布をQuery/Keyに加算し、境界・barb構造への感度を高める。
    Value・出力チャネル数はbottleneckの特徴マップと同一に保つ
    (MediSegUNetへdrop-inできるよう設計)。
    """

    def __init__(self, in_channels: int, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        # E = sigmoid(Conv3x3(入力画像)): 単一チャンネルのエッジ事前分布
        self.edge_conv = nn.Conv2d(in_channels, 1, kernel_size=3, padding=1)
        # E' = Conv1x1(interpolate(E)): bottleneckのチャンネル数へ射影
        self.edge_proj = nn.Conv2d(1, bottleneck_channels, kernel_size=1)
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor, original_image: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        edge_map = torch.sigmoid(self.edge_conv(original_image))
        edge_map = F.interpolate(edge_map, size=(h, w), mode="bilinear", align_corners=True)
        edge_prior = self.edge_proj(edge_map)  # (B, C, H, W)

        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        edge_flat = edge_prior.flatten(2).transpose(1, 2)  # (B, N, C)
        qk = x_flat + edge_flat  # Q = K = x + E'。位置エンコーディングは使わない(論文通り)

        attn_out, _ = self.mhsa(qk, qk, x_flat)  # V = x(エッジ由来の加算なし)
        attn_out = self.norm(attn_out + x_flat)  # 残差接続 + LayerNorm(論文に明記はないが標準的なTransformerブロックとして採用)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegEdgeAttUNet(nn.Module):
    """MediSegUNetのbottleneckにEdgeGuidedMHSAを挿入したdrop-in代替。

    入出力インターフェースはMediSegUNetと完全に同一(in_channels=3の
    crop、out_channels=1の確率マップ)。既存のrefiner学習・推論コードを
    変更せず、モデルクラスの差し替えだけで比較できるように設計している。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.edge_attn = EdgeGuidedMHSA(in_channels, bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.edge_attn(x5, x)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)


# ============================================================
# 5. Plain MHSA U-Net (matched-parameter ablation control, no edge prior)
# ============================================================
# 2026-08-31、report/main.tex Future Work item 1(matched-parameter
# ablation)のために追加。EdgeGuidedMHSAからedge_conv/edge_projを除いた
# だけの構成——Q=K=V=bottleneck特徴量の標準的なself-attentionで、
# edge priorをQ/Kへ加算する項だけがない。edge_conv/edge_projは
# 3(in_channels)*1*3*3+1 + 1*1024*1*1+1024 ≈ 2,076パラメータしかなく
# 全体(約35.2M)の0.006%程度なので、このクラスはMediSegEdgeAttUNetと
# ほぼ完全にパラメータ数が一致する("U-Net + MHSA, no positional
# encoding"に相当、EdgeAttNet論文のablationと同じ切り分け)。
# report/main.texが指摘してきた「edge priorそのものの効果か、単に
# attention容量が増えただけか」という交絡を分離するための唯一の差分。


class PlainMHSA(nn.Module):
    """edge priorなしの標準Multi-Head Self-Attention(bottleneck上)。

    EdgeGuidedMHSAと同じchannel数・head数・残差接続+LayerNormを保ちつつ、
    Q・Kへの edge_prior 加算だけを行わない(Q=K=V=bottleneck特徴量)。
    """

    def __init__(self, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        attn_out, _ = self.mhsa(x_flat, x_flat, x_flat)  # Q=K=V=x(edge priorなし)
        attn_out = self.norm(attn_out + x_flat)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegPlainMHSAUNet(nn.Module):
    """MediSegEdgeAttUNetからedge priorだけを除いた matched-parameter 対照群。

    入出力インターフェースはMediSegUNet/MediSegEdgeAttUNetと完全に同一。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.plain_attn = PlainMHSA(bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.plain_attn(x5)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)

"""Spine auxiliary-teacher target construction (train-time only).

Pre-registered design (SPINE_AUX_TEACHER_PREREG_2026-11-13.md): a small
head predicts a spine heatmap from shared P3 features; the target is the
annotation spine polyline rasterized into a one-cell-radius positive
band on the P3 grid. Positives/negatives get separately-averaged BCE
over valid cells only (missing-spine regions and padding excluded).
Inference uses only the H-alpha image; the aux head is removed.

All functions are pure-numpy so the whole target path is CPU-testable.
"""
import numpy as np


def _cell_coords(points: np.ndarray, img_h: int, img_w: int,
                 out_h: int, out_w: int) -> np.ndarray:
    """Image-space (x,y) points -> integer (cy,cx) cell indices, in-bounds only."""
    pts = np.asarray(points, dtype=np.float64).reshape(-1, 2)
    keep = (pts[:, 0] >= 0) & (pts[:, 0] < img_w) & (pts[:, 1] >= 0) & (pts[:, 1] < img_h)
    pts = pts[keep]
    cy = np.clip((pts[:, 1] * out_h / img_h).astype(np.int64), 0, out_h - 1)
    cx = np.clip((pts[:, 0] * out_w / img_w).astype(np.int64), 0, out_w - 1)
    return np.stack([cy, cx], axis=1)


def rasterize_polyline_band(points: np.ndarray, out_h: int, out_w: int,
                            img_h: int, img_w: int,
                            radius_cells: int = 1) -> np.ndarray:
    """Rasterize a spine polyline into a positive band on the feature grid.

    Segments are supersampled at <= half-cell spacing so no cell on the
    path is skipped; each stamped cell is then dilated by ``radius_cells``.
    Returns a bool array of shape (out_h, out_w).
    """
    pts = np.asarray(points, dtype=np.float64).reshape(-1, 2)
    band = np.zeros((out_h, out_w), dtype=bool)
    if len(pts) == 0:
        return band
    cell_h = img_h / out_h
    cell_w = img_w / out_w
    step = 0.5 * min(cell_h, cell_w)

    samples = [pts[0]]
    for p0, p1 in zip(pts[:-1], pts[1:]):
        dist = float(np.hypot(*(p1 - p0)))
        n = max(1, int(np.ceil(dist / step)))
        samples.extend(p0 + (p1 - p0) * t for t in np.linspace(0.0, 1.0, n + 1)[1:])
    cells = _cell_coords(np.asarray(samples), img_h, img_w, out_h, out_w)
    if len(cells):
        band[cells[:, 0], cells[:, 1]] = True
    return dilate_cells(band, radius_cells) if radius_cells > 0 else band


def dilate_cells(mask: np.ndarray, radius_cells: int) -> np.ndarray:
    """Chebyshev dilation of a bool grid by ``radius_cells`` cells (no wrap)."""
    return _dilate_cells_no_wrap(mask, radius_cells) if radius_cells > 0 else np.asarray(mask, bool)


def _dilate_cells_no_wrap(mask: np.ndarray, radius: int) -> np.ndarray:
    mask = np.asarray(mask, dtype=bool)
    h, w = mask.shape
    out = mask.copy()
    for _ in range(radius):
        padded = np.pad(out, 1, constant_values=False)
        grown = np.zeros_like(out)
        for dy in range(3):
            for dx in range(3):
                grown |= padded[dy:dy + h, dx:dx + w]
        out = grown
    return out


def transform_points_affine(points: np.ndarray, scale: tuple[float, float],
                            offset: tuple[float, float] = (0.0, 0.0)) -> np.ndarray:
    """Apply scale+translate to (x,y) points (letterbox / mosaic placement)."""
    pts = np.asarray(points, dtype=np.float64).reshape(-1, 2).copy()
    pts[:, 0] = pts[:, 0] * scale[0] + offset[0]
    pts[:, 1] = pts[:, 1] * scale[1] + offset[1]
    return pts


def flip_points_x(points: np.ndarray, img_w: int) -> np.ndarray:
    """Horizontal flip of (x,y) points, matching ``img[:, ::-1]``."""
    pts = np.asarray(points, dtype=np.float64).reshape(-1, 2).copy()
    pts[:, 0] = img_w - 1.0 - pts[:, 0]
    return pts


def padding_exclusion_mask(out_h: int, out_w: int, img_h: int, img_w: int,
                           pad: tuple[int, int, int, int] = (0, 0, 0, 0)) -> np.ndarray:
    """Bool (out_h,out_w) validity mask; False over cells inside padding.

    ``pad`` = (left, top, right, bottom) pixels of the *canvas* (post-
    letterbox image size img_w×img_h) that carry no real image content.
    A cell is invalid when its centre falls in a padded band.
    """
    left, top, right, bottom = pad
    valid = np.ones((out_h, out_w), dtype=bool)
    cy = (np.arange(out_h) + 0.5) * img_h / out_h
    cx = (np.arange(out_w) + 0.5) * img_w / out_w
    valid &= (cy[:, None] >= top) & (cy[:, None] < img_h - bottom)
    valid &= (cx[None, :] >= left) & (cx[None, :] < img_w - right)
    return valid


def build_spine_target(spines: list[np.ndarray],
                       valid_spine: list[bool] | None = None,
                       out_h: int = 160, out_w: int = 160,
                       img_h: int = 2048, img_w: int = 2048,
                       radius_cells: int = 1,
                       missing_regions: list[np.ndarray] | None = None,
                       valid_mask: np.ndarray | None = None) -> tuple[np.ndarray, np.ndarray]:
    """Build (positive_band, valid_mask) for one image.

    ``spines``: list of (N,2) image-space polylines, one per annotation.
    ``valid_spine``: per-annotation flag; entries with False contribute no
    positive cells *and* (via ``missing_regions``) may carve invalid cells.
    ``missing_regions``: bool (out_h,out_w) masks marking cells that must
    be excluded from supervision (e.g. the GT region of a missing-spine
    annotation). ``valid_mask``: precomputed validity (e.g. padding).
    """
    pos = np.zeros((out_h, out_w), dtype=bool)
    if valid_spine is None:
        valid_spine = [True] * len(spines)
    for spine, ok in zip(spines, valid_spine):
        if ok and len(np.asarray(spine).reshape(-1)) >= 4:
            pos |= rasterize_polyline_band(spine, out_h, out_w, img_h, img_w,
                                           radius_cells=radius_cells)
    if valid_mask is None:
        valid_mask = np.ones((out_h, out_w), dtype=bool)
    else:
        valid_mask = np.asarray(valid_mask, dtype=bool).copy()
    for region in missing_regions or []:
        valid_mask &= ~np.asarray(region, dtype=bool)
    return pos, valid_mask


def spine_aux_bce(logits: np.ndarray, pos: np.ndarray, valid: np.ndarray,
                  eps: float = 1e-7) -> tuple[float, float, float, int, int]:
    """Positive/negative separately-averaged BCE over valid cells.

    Returns (loss, pos_term, neg_term, n_pos, n_neg). Each term is the
    mean over its own class so the aux loss is decoupled from band
    sparsity; an absent class contributes 0.
    """
    logits = np.asarray(logits, dtype=np.float64)
    p = 1.0 / (1.0 + np.exp(-logits))
    pos = np.asarray(pos, dtype=bool)
    valid = np.asarray(valid, dtype=bool)
    pos_sel = pos & valid
    neg_sel = (~pos) & valid
    pos_term = float(-np.log(np.clip(p[pos_sel], eps, 1.0)).mean()) if pos_sel.any() else 0.0
    neg_term = float(-np.log(np.clip(1.0 - p[neg_sel], eps, 1.0)).mean()) if neg_sel.any() else 0.0
    return pos_term + neg_term, pos_term, neg_term, int(pos_sel.sum()), int(neg_sel.sum())


def cellize_mask_any(mask: np.ndarray, out_h: int, out_w: int) -> np.ndarray:
    """Downsample a bool image mask to a grid: cell positive iff any pixel.

    Uses an integral image so non-integer image/cell ratios (e.g. 2048 ->
    160 = 12.8 px/cell) are handled exactly; nearest-neighbour resize
    would erase thin structures and must not be used for this check.
    """
    mask = np.asarray(mask, dtype=bool)
    h, w = mask.shape
    ii = np.pad(mask.astype(np.int64).cumsum(0).cumsum(1), ((1, 0), (1, 0)))
    er = np.floor((np.arange(out_h) + 1) * h / out_h).astype(np.int64)
    ec = np.floor((np.arange(out_w) + 1) * w / out_w).astype(np.int64)
    sr = np.floor(np.arange(out_h) * h / out_h).astype(np.int64)
    sc = np.floor(np.arange(out_w) * w / out_w).astype(np.int64)
    sums = (ii[np.ix_(er, ec)] - ii[np.ix_(sr, ec)]
            - ii[np.ix_(er, sc)] + ii[np.ix_(sr, sc)])
    return sums > 0


def spine_in_mask_fraction(spine_band: np.ndarray, mask_band: np.ndarray,
                           dilate: int = 1) -> float:
    """Fraction of positive spine cells inside the (dilated) GT mask band.

    Pre-training consistency check: rasterize the GT union mask and the
    spine under the *same* geometric transform, then verify the spine
    band still lands inside the mask (audit-level expectation ~1.0).
    Returns 1.0 when there are no spine cells (nothing to contradict).
    """
    spine = np.asarray(spine_band, dtype=bool)
    mask = np.asarray(mask_band, dtype=bool)
    if dilate > 0:
        mask = _dilate_cells_no_wrap(mask, dilate)
    n = int(spine.sum())
    return float((spine & mask).sum() / n) if n else 1.0

"""Spine aux-teacher — ultralytics integration glue & trainer (train-time + Arm B+ rescue).

Design (prereg 2026-11-13): spine polylines travel through the standard
YOLO-seg label/transform pipeline as **pseudo-instances of a reserved
class** so every geometric transform (mosaic concat, RandomPerspective
``apply_segments`` affine, flips, letterbox) applies identical math to
spine and mask. Immediately before the final ``Format`` transform a
``SplitSpineSegments`` step removes the pseudo-instances and stores the
transformed polylines on ``labels['spine_segments_px']``; the detection
pipeline then sees only real instances (model stays nc=1).

Constraints verified against ultralytics 8.4.118–8.4.120 source:
- ``verify_image_label`` asserts ``max_cls < num_cls`` -> the dataset
  yaml for spine training must declare ``nc: 2`` ({0: filament,
  1: spine_aux}); ``single_cls`` MUST be ``False`` (else ``BaseDataset``
  overwrites ``cls=1`` to ``0`` before transforms), and the custom trainer
  forces ``model.nc = 1`` and ``model.names = {0: 'filament'}``.
- ``Format.apply_instances`` calls ``cls.squeeze(1)`` -> ``labels['cls']``
  after ``SplitSpineSegments`` must have 2D shape ``(N_keep, 1)``.
- ``YOLODataset.collate_fn`` uses ``zip(*values)``, producing a ``tuple``
  of length ``B`` for custom keys -> ``split_spine_batch`` accepts both
  ``list`` and ``tuple``.
- ``SpineAuxLoss.call`` computes BCE in ``float32`` via log-sigmoid so
  AMP ``float16`` never underflows ``1 - 1e-7 -> 1.0`` into ``NaN`` grads.
"""
from collections.abc import Sequence

import numpy as np

SPINE_CLS = 1  # reserved pseudo-class in the spine-augmented dataset yaml


def spine_label_row(spine_xy: np.ndarray, img_h: int, img_w: int) -> str:
    """Format one annotation spine as a YOLO-seg label row of SPINE_CLS.

    ``spine_xy``: (N,2) image-space polyline. Output is the normalized
    ``cls x1 y1 x2 y2 ...`` polygon-format row; a polyline is a valid
    (degenerate) polygon for the transform math we rely on.
    """
    pts = np.asarray(spine_xy, dtype=np.float64).reshape(-1, 2)
    if len(pts) < 2:
        raise ValueError("spine polyline needs >= 2 points")
    norm = pts.copy()
    norm[:, 0] = np.clip(norm[:, 0] / img_w, 0.0, 1.0)
    norm[:, 1] = np.clip(norm[:, 1] / img_h, 0.0, 1.0)
    flat = " ".join(f"{v:.6f}" for v in norm.reshape(-1))
    return f"{SPINE_CLS} {flat}"


def _ensure_min_three_points(pts: np.ndarray) -> np.ndarray:
    """Subdivide a 2-point segment by its midpoint so YOLO-seg sees >=7 tokens (not a 5-token bbox)."""
    pts = np.asarray(pts, dtype=np.float64).reshape(-1, 2)
    if len(pts) == 2:
        mid = 0.5 * (pts[0] + pts[1])
        return np.vstack([pts[0], mid, pts[1]])
    return pts


def emit_label_txt(
    instances: list[np.ndarray],
    spines: list[np.ndarray],
    img_h: int,
    img_w: int,
    real_cls: int = 0,
) -> str:
    """Emit a full label .txt: real instance polygons + spine pseudo-rows.

    ``instances``: list of (M,2) polygon point arrays (image coords).
    ``spines``: list of (N,2) spine polylines, same order as instances.
    2-point spines are midpoint-subdivided to 3 collinear points so
    Ultralytics ``verify_image_label`` never rejects them as 5-column boxes.
    """
    rows = []
    for poly in instances:
        pts = np.asarray(poly, dtype=np.float64).reshape(-1, 2)
        if len(pts) < 3:
            continue
        norm = pts.copy()
        norm[:, 0] = np.clip(norm[:, 0] / img_w, 0.0, 1.0)
        norm[:, 1] = np.clip(norm[:, 1] / img_h, 0.0, 1.0)
        rows.append(f"{real_cls} " + " ".join(f"{v:.6f}" for v in norm.reshape(-1)))
    for spine in spines:
        pts = np.asarray(spine, dtype=np.float64).reshape(-1, 2)
        if len(pts) >= 2:
            rows.append(spine_label_row(_ensure_min_three_points(pts), img_h, img_w))
    return "\n".join(rows) + ("\n" if rows else "")


def _pad_lrtb_from_labels(labels: dict) -> tuple[int, int, int, int]:
    """Extract (left, top, right, bottom) canvas padding pixels from LetterBox metadata."""
    ratio_pad = labels.get("ratio_pad")
    if not isinstance(ratio_pad, (list, tuple)) or len(ratio_pad) < 2:
        return (0, 0, 0, 0)
    pad_wh = ratio_pad[1]
    if not isinstance(pad_wh, (list, tuple)) or len(pad_wh) < 2:
        return (0, 0, 0, 0)
    dw, dh = float(pad_wh[0]), float(pad_wh[1])
    left = max(0, round(dw - 0.1))
    right = max(0, round(dw + 0.1))
    top = max(0, round(dh - 0.1))
    bottom = max(0, round(dh + 0.1))
    return (left, top, right, bottom)


class SplitSpineSegments:
    """Pre-Format transform: move SPINE_CLS instances to spine_segments_px.

    Sits last in the geometric Compose (inserted just before ``Format``).
    Preserves ``cls`` as 2D ``(N_keep, 1)`` when input is 2D or ``img`` is
    present (required by ``Format.apply_instances`` ``cls.squeeze(1)``), and
    converts normalized segments to canvas pixel coordinates when
    ``augment=False`` (e.g. validation or ``close_mosaic`` epochs).
    """

    def __init__(self, spine_cls: int = SPINE_CLS):
        self.spine_cls = spine_cls

    def __call__(self, labels: dict) -> dict:
        instances = labels.get("instances")
        cls = labels.get("cls")
        labels["spine_pad_lrtb"] = _pad_lrtb_from_labels(labels)
        if instances is None or cls is None:
            labels["spine_segments_px"] = []
            return labels
        cls_arr = np.asarray(cls)
        keep_2d = cls_arr.ndim == 2 or "img" in labels
        flat_cls = cls_arr.reshape(-1)
        is_spine = flat_cls == self.spine_cls
        segs = np.asarray(instances.segments)
        if segs.ndim == 3 and "img" in labels and getattr(instances, "normalized", False):
            h, w = labels["img"].shape[:2]
            scale = np.array([float(w), float(h)], dtype=segs.dtype)
            segs = segs * scale
        labels["spine_segments_px"] = (
            [segs[i] for i in np.flatnonzero(is_spine)] if segs.ndim == 3 else []
        )
        keep = ~is_spine
        labels["instances"] = instances[keep]
        kept_cls = flat_cls[keep]
        labels["cls"] = kept_cls.reshape(-1, 1) if keep_2d else kept_cls
        return labels


def split_spine_batch(batch: dict, spine_key: str = "spine_segments_px") -> list[list[np.ndarray]]:
    """Collect per-image spine polyline lists from a collated batch.

    ``YOLODataset.collate_fn`` aggregates unknown keys via ``zip(*values)``,
    producing a ``tuple`` of length ``B``; direct callers may pass a ``list``.
    """
    value = batch.get(spine_key, ())
    if isinstance(value, (list, tuple)):
        return [list(v) if isinstance(v, (list, tuple)) else [v] for v in value]
    return [[value]] if len(value) else [[]]


def split_pad_batch(
    batch: dict,
    batch_size: int,
    pad_key: str = "spine_pad_lrtb",
) -> list[tuple[int, int, int, int]]:
    """Extract per-image (left, top, right, bottom) padding tuples from a batch."""
    raw = batch.get(pad_key)
    if isinstance(raw, (list, tuple)) and len(raw) == batch_size:
        out: list[tuple[int, int, int, int]] = []
        for item in raw:
            if isinstance(item, (list, tuple)) and len(item) == 4:
                out.append((int(item[0]), int(item[1]), int(item[2]), int(item[3])))
            else:
                out.append((0, 0, 0, 0))
        return out
    return [(0, 0, 0, 0)] * batch_size


def build_p3_targets(
    spine_lists: list[list[np.ndarray]],
    batch_shape_hw: tuple[int, int],
    grid: int | tuple[int, int] = 160,
    radius_cells: int = 1,
    pads: Sequence[tuple[int, int, int, int]] | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """Rasterize each batch image's spine polylines to (pos, valid) grids."""

    out_h, out_w = (grid, grid) if isinstance(grid, int) else (int(grid[0]), int(grid[1]))
    h, w = int(batch_shape_hw[0]), int(batch_shape_hw[1])
    bsz = len(spine_lists)
    pos = np.zeros((bsz, out_h, out_w), dtype=bool)
    valid = np.ones_like(pos)
    for i, spines in enumerate(spine_lists):
        pad = pads[i] if pads is not None and i < len(pads) else (0, 0, 0, 0)
        vmask = (
            padding_exclusion_mask(out_h, out_w, h, w, pad=pad)
            if any(pad)
            else None
        )
        p, v = build_spine_target(
            spines,
            out_h=out_h,
            out_w=out_w,
            img_h=h,
            img_w=w,
            radius_cells=radius_cells,
            valid_mask=vmask,
        )
        pos[i], valid[i] = p, v
    return pos, valid


class SpineAuxLoss:
    """AMP-safe torch implementation of positive/negative separately-averaged BCE."""

    @staticmethod
    def call(logits, pos, valid, eps: float = 1e-7):
        import torch
        import torch.nn.functional as F

        del eps  # log-sigmoid BCE is exact and avoids float16 clamp rounding to 1.0
        logits32 = logits.float()
        pos_t = torch.as_tensor(pos, dtype=torch.bool, device=logits.device)
        valid_t = torch.as_tensor(valid, dtype=torch.bool, device=logits.device)
        if pos_t.ndim == 3 and logits32.ndim == 4:
            pos_t = pos_t.unsqueeze(1)
            valid_t = valid_t.unsqueeze(1)
        pos_sel = pos_t & valid_t
        neg_sel = (~pos_t) & valid_t
        zero = logits32.sum() * 0.0
        pos_term = (
            F.binary_cross_entropy_with_logits(
                logits32[pos_sel], torch.ones_like(logits32[pos_sel]), reduction="mean"
            )
            if pos_sel.any()
            else zero
        )
        neg_term = (
            F.binary_cross_entropy_with_logits(
                logits32[neg_sel], torch.zeros_like(logits32[neg_sel]), reduction="mean"
            )
            if neg_sel.any()
            else zero
        )
        return pos_term + neg_term


def compute_spine_ridge_score(
    mask: np.ndarray,
    spine_prob: np.ndarray,
    *,
    top_quantile: float = 0.75,
) -> float:
    """Mean spine heatmap probability along the top-quantile ridge inside ``mask``.

    Maps pixel coordinates directly onto the P3 ``spine_prob`` grid ``(gh, gw)``
    without upsampling ``spine_prob`` to ``2048x2048``.
    """
    ys, xs = np.nonzero(mask)
    if len(xs) == 0:
        return 0.0
    h, w = mask.shape
    gh, gw = spine_prob.shape
    cy = np.clip((ys.astype(np.int64) * gh) // h, 0, gh - 1)
    cx = np.clip((xs.astype(np.int64) * gw) // w, 0, gw - 1)
    vals = spine_prob[cy, cx].astype(np.float64)
    if len(vals) <= 4 or top_quantile <= 0.0:
        return float(vals.mean())
    thr = float(np.quantile(vals, top_quantile))
    top_vals = vals[vals >= thr]
    return float(top_vals.mean()) if len(top_vals) else float(vals.mean())


def selective_spine_rescue(
    conf: Sequence[float] | np.ndarray,
    areas: Sequence[float] | np.ndarray,
    spine_scores: Sequence[float] | np.ndarray,
    *,
    primary_conf: float = 0.26,
    rescue_floor: float = 0.14,
    min_spine: float = 0.65,
    min_area: int = 20,
) -> list[int]:
    """Keep all primary proposals plus low-conf proposals with strong P3 spine ridge support."""
    conf_arr = np.asarray(conf, dtype=np.float64)
    area_arr = np.asarray(areas, dtype=np.float64)
    spine_arr = np.asarray(spine_scores, dtype=np.float64)
    if not (len(conf_arr) == len(area_arr) == len(spine_arr)):
        raise ValueError("conf, areas, and spine_scores must have equal length")
    kept: list[int] = []
    for i in range(len(conf_arr)):
        if area_arr[i] < min_area:
            continue
        if conf_arr[i] >= primary_conf or (
            conf_arr[i] >= rescue_floor and spine_arr[i] >= min_spine
        ):
            kept.append(i)
    return kept


def build_spine_trainer_classes(aux_weight: float = 0.0, radius_cells: int = 1):
    """Build Ultralytics Dataset, Loss, and SegmentationTrainer classes for Spine A/B.

    Both Arm A (``aux_weight=0.0``) and Arm B (``aux_weight>0.0``) can share the
    exact same dataset/transform/trainer path so the ONLY difference between
    Arm A and Arm B is the P3 ``1x1 conv`` spine auxiliary supervision.
    """
    from copy import copy

    from torch import nn
    from ultralytics.data.dataset import YOLODataset
    from ultralytics.models.yolo.segment import SegmentationTrainer
    from ultralytics.nn.tasks import SegmentationModel
    from ultralytics.utils import DEFAULT_CFG, RANK
    from ultralytics.utils.loss import v8SegmentationLoss
    from ultralytics.utils.torch_utils import unwrap_model

    class SpineYOLODataset(YOLODataset):
        def build_transforms(self, hyp=None):
            transforms = super().build_transforms(hyp)
            transforms.insert(-1, SplitSpineSegments(spine_cls=SPINE_CLS))
            return transforms

    class SpineSegmentationLoss(v8SegmentationLoss):
        def __init__(self, model: nn.Module, tal_topk: int = 10):
            super().__init__(model, tal_topk=tal_topk)
            self.spine_head = getattr(model, "spine_head", None)
            self.aux_weight = float(aux_weight)
            self.radius_cells = int(radius_cells)

        def loss(self, preds: dict, batch: dict):
            loss_vec, loss_items = super().loss(preds, batch)
            if (
                self.aux_weight > 0.0
                and self.spine_head is not None
                and "spine_segments_px" in batch
            ):
                p3_feat = preds["feats"][0]
                aux_logits = self.spine_head(p3_feat)
                bsz = int(aux_logits.shape[0])
                spine_lists = split_spine_batch(batch)
                if len(spine_lists) == bsz:
                    pads = split_pad_batch(batch, bsz)
                    pos, valid = build_p3_targets(
                        spine_lists,
                        batch["img"].shape[-2:],
                        grid=aux_logits.shape[-2:],
                        radius_cells=self.radius_cells,
                        pads=pads,
                    )
                    aux_raw = SpineAuxLoss.call(aux_logits, pos, valid)
                    weighted = self.aux_weight * aux_raw
                    loss_vec[4] = loss_vec[4] + weighted * bsz
                    loss_items["sem_loss"] = (loss_items["sem_loss"] + weighted).detach()
            return loss_vec, loss_items

    class SpineSegmentationTrainer(SegmentationTrainer):
        def __init__(self, cfg=DEFAULT_CFG, overrides=None, _callbacks=None):
            overrides = dict(overrides or {})
            # Never allow BaseDataset.update_labels to overwrite SPINE_CLS=1 -> 0
            overrides["single_cls"] = False
            super().__init__(cfg=cfg, overrides=overrides, _callbacks=_callbacks)

        def get_model(self, cfg=None, weights=None, verbose: bool = True):
            # Force single real output class (0: filament) regardless of data.yaml nc=2
            orig_names = self.data.get("names")
            self.data = {**self.data, "names": {0: "filament"}}
            try:
                model = self.set_model_names_for_load(
                    SegmentationModel(
                        cfg,
                        nc=1,
                        ch=self.data.get("channels", 3),
                        verbose=verbose and RANK == -1,
                    )
                )
                if weights:
                    model.load(weights)
            finally:
                if orig_names is not None:
                    self.data["names"] = orig_names
            model.nc = 1
            model.names = {0: "filament"}
            if aux_weight > 0.0:
                seg_head = model.model[-1]
                p3_ch = int(seg_head.cv2[0][0].conv.in_channels)
                model.spine_head = nn.Conv2d(p3_ch, 1, kernel_size=1)
            return model

        def set_model_attributes(self):
            super().set_model_attributes()
            umodel = unwrap_model(self.model)
            umodel.nc = 1
            umodel.names = {0: "filament"}
            umodel.criterion = SpineSegmentationLoss(umodel)

        def get_class_counts(self):
            classes = np.concatenate(
                [lb["cls"].reshape(-1) for lb in self.train_loader.dataset.labels], 0
            )
            real = classes[classes == 0].astype(int)
            return np.bincount(real, minlength=1).astype(np.float32)

        def build_dataset(self, img_path: str, mode: str = "train", batch: int | None = None):
            gs = max(int(unwrap_model(self.model).stride.max()), 32)
            return SpineYOLODataset(
                img_path=img_path,
                imgsz=self.args.imgsz,
                batch_size=batch,
                augment=mode == "train",
                hyp=self.args,
                rect=self.args.rect or mode == "val",
                cache=self.args.cache or None,
                single_cls=False,
                stride=gs,
                pad=0.0 if mode == "train" else 0.5,
                prefix=f"{mode}: ",
                task=self.args.task,
                classes=self.args.classes,
                data=self.data,
                fraction=self.args.fraction if mode == "train" else 1.0,
            )

        def get_validator(self):
            val_args = copy(self.args)
            val_args.single_cls = True
            from ultralytics.models import yolo
            return yolo.segment.SegmentationValidator(
                self.test_loader,
                save_dir=self.save_dir,
                args=val_args,
                _callbacks=self.callbacks,
            )

    return SpineYOLODataset, SpineSegmentationLoss, SpineSegmentationTrainer


def extract_p3_spine_prob(yolo_obj, image_path: str, *, imgsz: int = 1280, conf: float = 0.02, iou: float = 0.55):
    """Run standard ``YOLO.predict`` while capturing P3 ``spine_head`` probability map ``(H_p3, W_p3)``.

    Returns ``(result, spine_prob_or_none)``. If ``yolo_obj.model`` has no ``spine_head``
    (e.g. Arm A), ``spine_prob_or_none`` is ``None`` and prediction is identical to standard YOLO.
    """
    import torch

    net = yolo_obj.model
    spine_head = getattr(net, "spine_head", None)
    if spine_head is None:
        res = yolo_obj.predict(
            str(image_path),
            imgsz=imgsz,
            conf=conf,
            iou=iou,
            agnostic_nms=True,
            retina_masks=True,
            verbose=False,
        )[0]
        return res, None

    p3_layer_idx = int(net.model[-1].f[0])
    captured: list[torch.Tensor] = []

    def _hook(_module, _inp, out):
        captured.append(out.detach())

    handle = net.model[p3_layer_idx].register_forward_hook(_hook)
    try:
        res = yolo_obj.predict(
            str(image_path),
            imgsz=imgsz,
            conf=conf,
            iou=iou,
            agnostic_nms=True,
            retina_masks=True,
            verbose=False,
        )[0]
        if not captured:
            return res, None
        with torch.no_grad():
            feat = captured[-1]
            spine_head = spine_head.to(device=feat.device, dtype=feat.dtype).eval()
            prob = torch.sigmoid(spine_head(feat).float())[0, 0].cpu().numpy()
        return res, prob
    finally:
        handle.remove()

"""320x320 Spine-guided EdgeAttUNet refiner target, mass-preserving loss, and model wrapper.

Implements Idea 5 (v129): applies organizer-approved (Topic 755241) human ``spine``
centerline supervision at full ``320x320`` crop resolution inside ``MediSegEdgeAttUNet``
while keeping test-time inference 100% architecture-identical (vanilla ``3 -> 1``
``MediSegEdgeAttUNet`` + ``shrink_mask`` inside the YOLO proposal).
"""
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn


def crop_spine_target(
    spine_xy: np.ndarray | list,
    box: tuple[int, int, int, int],
    *,
    crop_size: int = 320,
    radius_px: int = 2,
    gt_crop_mask: np.ndarray | None = None,
) -> np.ndarray:
    """Rasterize an image-space ``spine`` polyline onto a ``(crop_size, crop_size)`` bool mask.

    If ``gt_crop_mask`` is provided, the rasterized spine band is intersected with
    ``gt_crop_mask > 0`` so spine positives never spill outside the ground-truth mask.
    """
    out = np.zeros((crop_size, crop_size), dtype=np.uint8)
    pts = np.asarray(spine_xy, dtype=np.float64).reshape(-1, 2)
    x0, y0, x1, y1 = map(float, box)
    bw = max(x1 - x0, 1.0)
    bh = max(y1 - y0, 1.0)
    if len(pts) >= 2:
        mapped = np.empty_like(pts)
        mapped[:, 0] = (pts[:, 0] - x0) * (crop_size / bw)
        mapped[:, 1] = (pts[:, 1] - y0) * (crop_size / bh)
        ipts = np.round(mapped).astype(np.int32).reshape(-1, 1, 2)
        thickness = max(1, 2 * int(radius_px) + 1)
        cv2.polylines(out, [ipts], isClosed=False, color=1, thickness=thickness, lineType=cv2.LINE_8)
    band = out > 0
    if gt_crop_mask is not None:
        gt_bool = np.asarray(gt_crop_mask) > 0
        clipped = band & gt_bool
        if clipped.any():
            band = clipped
        else:
            band = gt_bool & band
    return band


def mass_preserving_spine_weights(
    targets: torch.Tensor,
    spine_targets: torch.Tensor,
    *,
    gamma: float = 1.5,
) -> torch.Tensor:
    """Compute per-pixel BCE weights that upweight spine pixels while preserving foreground mass.

    For each sample ``b`` with foreground mask ``M = (targets > 0.5)`` and spine band
    ``S = (spine_targets > 0.5) & M``:
      - Background pixels ``~M`` get weight ``1.0``.
      - Foreground pixels ``M`` get ``(|M| / (|M| + gamma * |S|)) * (1 + gamma * S)`` so
        ``sum_{x in M} w(x) == |M|`` identically, preventing any shift of the ``0.45``
        inference probability threshold.
    """
    fg = targets > 0.5
    if gamma <= 0.0:
        return torch.ones_like(targets, dtype=torch.float32)
    sp = (spine_targets > 0.5) & fg
    fg_f = fg.float()
    sp_f = sp.float()
    fg_sum = fg_f.flatten(1).sum(dim=1).view(-1, 1, 1, 1)
    sp_sum = sp_f.flatten(1).sum(dim=1).view(-1, 1, 1, 1)
    scale = torch.where(
        fg_sum > 0.0,
        fg_sum / (fg_sum + float(gamma) * sp_sum).clamp(min=1e-6),
        torch.ones_like(fg_sum),
    )
    fg_weights = scale * (1.0 + float(gamma) * sp_f)
    return torch.where(fg, fg_weights, torch.ones_like(fg_weights))


def _boundary_map(t: torch.Tensor) -> torch.Tensor:
    dil = F.max_pool2d(t, 5, stride=1, padding=2)
    ero = -F.max_pool2d(-t, 5, stride=1, padding=2)
    return torch.clamp(dil - ero, 0.0, 1.0)


def spine_refiner_loss(
    mask_logits: torch.Tensor,
    targets: torch.Tensor,
    *,
    spine_logits: torch.Tensor | None = None,
    spine_targets: torch.Tensor | None = None,
    aux_weight: float = 0.0,
    spine_gamma: float = 0.0,
) -> torch.Tensor:
    """AMP-safe EdgeAttUNet loss with optional decoder spine aux head and mass-preserving BCE."""
    logits32 = mask_logits.float()
    targets32 = targets.float()
    probs = torch.sigmoid(logits32)
    inter = (probs * targets32).sum((1, 2, 3))
    denom = probs.sum((1, 2, 3)) + targets32.sum((1, 2, 3))
    sdice = (1.0 - (2.0 * inter + 1.0) / (denom + 1.0)).mean()

    if spine_gamma > 0.0 and spine_targets is not None:
        pix_w = mass_preserving_spine_weights(targets32, spine_targets.float(), gamma=spine_gamma)
        bce_map = F.binary_cross_entropy_with_logits(logits32, targets32, reduction="none")
        bce = (bce_map * pix_w).mean()
    else:
        bce = F.binary_cross_entropy_with_logits(logits32, targets32)

    region = 0.35 * bce + 0.65 * sdice
    edge = F.l1_loss(_boundary_map(probs), _boundary_map(targets32))
    total = region + 0.15 * edge

    if aux_weight > 0.0 and spine_logits is not None and spine_targets is not None:
        sp_bool = spine_targets > 0.5
        valid = torch.ones_like(sp_bool, dtype=torch.bool)
        aux_loss = SpineAuxLoss.call(spine_logits, sp_bool, valid)
        total = total + float(aux_weight) * aux_loss
    return total


class SpineGuidedEdgeAttUNet(MediSegEdgeAttUNet):
    """MediSegEdgeAttUNet with a train-time-only 1x1 ``spine_head`` on decoder ``up4`` features.

    ``export_base_state_dict()`` strips ``spine_head.*`` keys so checkpoints load with
    ``strict=True`` into vanilla ``MediSegEdgeAttUNet(3, 1, False)``.
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__(
            in_channels=in_channels,
            out_channels=out_channels,
            bilinear=bilinear,
            num_heads=num_heads,
        )
        self.spine_head = nn.Conv2d(64, 1, kernel_size=1)

    def forward_with_aux(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.edge_attn(x5, x)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        dec = self.up4(out, x1)
        return self.outc(dec), self.spine_head(dec)

    def export_base_state_dict(self) -> dict[str, torch.Tensor]:
        return {
            k: v.detach().cpu().clone()
            for k, v in self.state_dict().items()
            if not k.startswith("spine_head.")
        }


In [ ]:
# [3/6] COCO 読込・台帳 SHA256 検証・分割固定・ローカル SSD ステージング
PINNED_EXCLUSIONS_CSV = """file_name,time_block_role,fold,exclusion_reason
20161229133734Ch.jpeg,development,0,within_5d_of_locked_eval
20161231133734Ch.jpeg,development,1,within_5d_of_locked_eval
20170102024454Uh.jpeg,locked_evaluation,1,within_5d_of_development
20170103030554Uh.jpeg,locked_evaluation,1,within_5d_of_development
"""

def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

TRAIN_JSON = next(Path(CFG.DATA_CACHE).rglob("MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
coco = json.loads(TRAIN_JSON.read_text(encoding="utf-8"))
images = {item["id"]: item for item in coco["images"]}
annotations = defaultdict(list)
for ann in coco["annotations"]:
    annotations[ann["image_id"]].append(ann)
file_to_ids = defaultdict(list)
for image_id, info in images.items():
    file_to_ids[info["file_name"]].append(image_id)
train_files = sorted(file_to_ids)

folds_path = CFG.ARTIFACT_ROOT / "grouped_folds.csv"
folds_sha = file_sha256(folds_path)
if folds_sha != CFG.GROUPED_FOLDS_SHA256:
    raise ValueError(f"grouped_folds.csv SHA256 mismatch: {folds_sha}")
fold_df = pd.read_csv(folds_path)

tb_path = CFG.ARTIFACT_ROOT / "time_block_assignment.csv"
if tb_path.exists():
    tb_sha = file_sha256(tb_path)
    if tb_sha != CFG.TIME_BLOCK_SHA256:
        raise ValueError(f"time_block_assignment.csv SHA256 mismatch: {tb_sha}")
    tb_df = pd.read_csv(tb_path)
else:
    tb_df = fold_df.copy()
    tb_df["year"] = tb_df["file_name"].str.slice(0, 4).astype(int)
    tb_df["time_block_role"] = np.where(tb_df["year"] <= 2016, "development", "locked_evaluation")
    tb_sha = "derived_year_le_2016"

excl_path = CFG.ARTIFACT_ROOT / "spine_aux_prereg_exclusions_2026-11-13.csv"
if excl_path.exists():
    excl_sha = file_sha256(excl_path)
    if excl_sha != CFG.EXCLUSIONS_SHA256:
        raise ValueError(f"exclusions CSV SHA256 mismatch: {excl_sha}")
    excl_df = pd.read_csv(excl_path)
else:
    raw_bytes = PINNED_EXCLUSIONS_CSV.encode("utf-8")
    excl_df = pd.read_csv(io.StringIO(PINNED_EXCLUSIONS_CSV))
    excl_sha = hashlib.sha256(raw_bytes).hexdigest()

excluded_set = set(excl_df["file_name"].tolist())
role_of_file = dict(zip(tb_df["file_name"], tb_df["time_block_role"]))
fold_of_file = dict(zip(fold_df["file_name"], fold_df["fold"].astype(int)))

if CFG.SPLIT_MODE == "pilot":
    fit_files = [f for f in train_files if role_of_file[f] == "development" and fold_of_file[f] in (0, 1) and f not in excluded_set]
    valid_files = [f for f in train_files if role_of_file[f] == "development" and fold_of_file[f] == 2 and f not in excluded_set]
    assert len(fit_files) == 365 and len(valid_files) == 180, (len(fit_files), len(valid_files))
    assert not any(role_of_file[f] == "locked_evaluation" for f in fit_files + valid_files)
elif CFG.SPLIT_MODE == "temporal_confirm":
    fit_files = [f for f in train_files if role_of_file[f] == "development" and f not in excluded_set]
    valid_files = [f for f in train_files if role_of_file[f] == "locked_evaluation"]
    assert len(fit_files) == 545 and len(valid_files) == 160, (len(fit_files), len(valid_files))
else:
    raise ValueError(f"Unknown SPLIT_MODE: {CFG.SPLIT_MODE}")
assert set(fit_files).isdisjoint(valid_files), "fit/eval overlap detected!"

for name in tqdm(sorted(set(fit_files) | set(valid_files)), desc="前処理画像をローカル SSD へコピー"):
    dst = CFG.PREPROC_DIR / name
    if not dst.exists():
        shutil.copy2(CFG.PREPROC_CACHE / name, dst)

def to_rle(annotation, height, width):
    seg = annotation["segmentation"]
    if isinstance(seg, list):
        return mask_utils.merge(mask_utils.frPyObjects(seg, height, width))
    if isinstance(seg.get("counts"), list):
        return mask_utils.frPyObjects(seg, height, width)
    rle = dict(seg)
    if isinstance(rle["counts"], str):
        rle["counts"] = rle["counts"].encode("utf-8")
    return rle

gt_rles, gt_areas = {}, {}
for image_id, info in images.items():
    h, w = int(info["height"]), int(info["width"])
    r_list, a_list = [], []
    for a in annotations[image_id]:
        if a.get("iscrowd", 0):
            continue
        r = to_rle(a, h, w)
        r_list.append(r)
        a_list.append(float(a.get("area", mask_utils.area(r))))
    gt_rles[image_id] = r_list
    gt_areas[image_id] = np.asarray(a_list, dtype=np.float64)

print({"split_mode": CFG.SPLIT_MODE, "fit_obs": len(fit_files), "eval_obs": len(valid_files),
       "folds_sha": folds_sha[:12], "tb_sha": tb_sha[:12], "excl_sha": excl_sha[:12]})


In [ ]:
# [4/6] SpineCropDataset 幾何同期検査 + 同一初期重み・同一乱数による 3 Arm 厳密対照学習
def square_box_from_bbox(bbox, height, width, context=CFG.CROP_CONTEXT, minimum=96):
    x, y, w, h = map(float, bbox)
    cx, cy = x + w / 2, y + h / 2
    side = int(math.ceil(max(w, h) * context))
    side = min(max(side, minimum), min(height, width))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > width: x0 -= x1 - width; x1 = width
    if y1 > height: y0 -= y1 - height; y1 = height
    return max(0, x0), max(0, y0), min(width, x1), min(height, y1)

class SpineCropDataset(torch.utils.data.Dataset):
    def __init__(self, obs_files, augment=True):
        self.augment = augment
        self.records = []
        for fname in obs_files:
            for iid in file_to_ids[fname]:
                info = images[iid]
                h, w = int(info["height"]), int(info["width"])
                valid_anns = [a for a in annotations[iid] if not a.get("iscrowd", 0)]
                for ann, rle in zip(valid_anns, gt_rles[iid]):
                    box = square_box_from_bbox(ann["bbox"], h, w)
                    sp = ann.get("spine", [])
                    sp_xy = np.asarray(sp, dtype=np.float64).reshape(-1, 2) if len(sp) >= 4 else np.zeros((0, 2), dtype=np.float64)
                    self.records.append((fname, rle, box, sp_xy))

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        fname, rle, box, sp_xy = self.records[idx]
        x0, y0, x1, y1 = box
        img = cv2.imread(str(CFG.PREPROC_DIR / fname), cv2.IMREAD_COLOR)
        mask = mask_utils.decode(rle).astype(np.uint8)
        crop = cv2.resize(img[y0:y1, x0:x1], (CFG.CROP_SIZE, CFG.CROP_SIZE), interpolation=cv2.INTER_LINEAR)
        mcrop = (cv2.resize(mask[y0:y1, x0:x1], (CFG.CROP_SIZE, CFG.CROP_SIZE), interpolation=cv2.INTER_NEAREST) > 0).astype(np.float32)
        scrop = crop_spine_target(
            sp_xy, box, crop_size=CFG.CROP_SIZE, radius_px=CFG.SPINE_RADIUS_PX, gt_crop_mask=mcrop
        ).astype(np.float32)

        if self.augment:
            k = random.randint(0, 3)
            if k:
                crop = np.rot90(crop, k).copy()
                mcrop = np.rot90(mcrop, k).copy()
                scrop = np.rot90(scrop, k).copy()
            if random.random() < 0.5:
                crop = np.fliplr(crop).copy()
                mcrop = np.fliplr(mcrop).copy()
                scrop = np.fliplr(scrop).copy()
            if random.random() < 0.5:
                crop = np.flipud(crop).copy()
                mcrop = np.flipud(mcrop).copy()
                scrop = np.flipud(scrop).copy()

        x_t = torch.from_numpy(crop.transpose(2, 0, 1).copy()).float() / 255.0
        if self.augment:
            for ch in (0, 1):
                if random.random() < 0.5:
                    g = random.uniform(0.85, 1.15)
                    x_t[ch] = torch.clamp(x_t[ch], 1e-6, 1.0).pow(g)
        return x_t, torch.from_numpy(mcrop[None].copy()).float(), torch.from_numpy(scrop[None].copy()).float()

rng_split = np.random.default_rng(CFG.SEED)
perm_fit = list(fit_files)
rng_split.shuffle(perm_fit)
n_hold = max(20, int(0.10 * len(perm_fit)))
ref_val_files, ref_tr_files = perm_fit[:n_hold], perm_fit[n_hold:]

tr_ds = SpineCropDataset(ref_tr_files, augment=True)
va_ds = SpineCropDataset(ref_val_files, augment=False)

# 事前幾何検査: 拡張後サンプル 48 件で spine ターゲットが 100% GT マスク内部にあることを確認
_nonempty_sp = 0
for _i in range(min(48, len(tr_ds))):
    _, _m, _s = tr_ds[_i]
    assert bool(((_s > 0.5) <= (_m > 0.5)).all()), f"Spine spilled outside GT crop at idx={_i}"
    _nonempty_sp += int((_s > 0.5).any())
print(f"[Crop Spine Sanity] tr_crops={len(tr_ds)}, va_crops={len(va_ds)}, checked=48, nonempty_spines={_nonempty_sp}/48 (100% inside GT)")

# 全 Arm 共通の初期重みを生成し、初期値分散を完全に排除
torch.manual_seed(CFG.SEED)
_init_net = SpineGuidedEdgeAttUNet(3, 1, False)
SHARED_INIT_SD = {k: v.detach().cpu().clone() for k, v in _init_net.state_dict().items()}
del _init_net

def _seed_worker(worker_id):
    s = CFG.SEED + worker_id
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)

def train_refiner_arm(arm_name: str, aux_weight: float, spine_gamma: float):
    ckpt_path = CFG.EXP_DIR / f"{arm_name}_{CFG.SPLIT_MODE}_fit{len(fit_files)}.pt"
    done_path = CFG.EXP_DIR / f"{arm_name}_{CFG.SPLIT_MODE}_fit{len(fit_files)}.DONE"
    if done_path.exists() and ckpt_path.exists():
        ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        print(f"再利用: {arm_name} ({ckpt_path.name}, best_dice={ck.get('best_dice', -1):.4f})")
        return ckpt_path, {"reused": True, "best_dice": float(ck.get("best_dice", -1.0)), "last_dice": float(ck.get("last_dice", -1.0))}

    random.seed(CFG.SEED); np.random.seed(CFG.SEED); torch.manual_seed(CFG.SEED); torch.cuda.manual_seed_all(CFG.SEED)
    gen = torch.Generator()
    gen.manual_seed(CFG.SEED)
    tr_loader = torch.utils.data.DataLoader(
        tr_ds, batch_size=CFG.REFINER_BATCH, shuffle=True, num_workers=2,
        pin_memory=True, worker_init_fn=_seed_worker, generator=gen,
    )
    va_loader = torch.utils.data.DataLoader(
        va_ds, batch_size=CFG.REFINER_BATCH, shuffle=False, num_workers=2, pin_memory=True,
    )

    model = SpineGuidedEdgeAttUNet(3, 1, False).cuda()
    model.load_state_dict(SHARED_INIT_SD, strict=True)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.REFINER_LR, weight_decay=CFG.REFINER_WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG.REFINER_EPOCHS)
    scaler = torch.amp.GradScaler("cuda")

    best_dice, best_sd = -1.0, None
    history = []
    t0 = time.time()
    for ep in range(1, CFG.REFINER_EPOCHS + 1):
        model.train()
        loss_sum, n_batches = 0.0, 0
        for xb, yb, sb in tqdm(tr_loader, desc=f"{arm_name} ep {ep}/{CFG.REFINER_EPOCHS}", leave=False):
            xb = xb.cuda(non_blocking=True)
            yb = yb.cuda(non_blocking=True)
            sb = sb.cuda(non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                mask_logits, spine_logits = model.forward_with_aux(xb)
                loss = spine_refiner_loss(
                    mask_logits, yb,
                    spine_logits=spine_logits,
                    spine_targets=sb,
                    aux_weight=aux_weight,
                    spine_gamma=spine_gamma,
                )
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            loss_sum += float(loss.detach().item()); n_batches += 1
        sched.step()

        model.eval()
        d_sum, d_cnt = 0.0, 0
        with torch.no_grad():
            for xb, yb, _ in va_loader:
                xb = xb.cuda(non_blocking=True)
                yb = yb.cuda(non_blocking=True).bool()
                pr = model(xb).sigmoid() >= 0.5
                inter = (pr & yb).sum((1, 2, 3)).float()
                den = pr.sum((1, 2, 3)) + yb.sum((1, 2, 3))
                d_sum += ((2.0 * inter + 1.0) / (den + 1.0)).sum().item()
                d_cnt += len(xb)
        dice = d_sum / max(d_cnt, 1)
        history.append({"ep": ep, "train_loss": round(loss_sum / max(n_batches, 1), 5), "fit_holdout_dice": round(dice, 5)})
        print(f"  [{arm_name}] ep={ep:02d} loss={loss_sum / max(n_batches, 1):.4f} fit-holdout dice={dice:.4f}")
        if dice > best_dice:
            best_dice = dice
            best_sd = model.export_base_state_dict()

    last_sd = model.export_base_state_dict()
    last_dice = history[-1]["fit_holdout_dice"]
    # Exported state_dict が純正 MediSegEdgeAttUNet に strict=True で読めることを即座に保証
    _vanilla = MediSegEdgeAttUNet(3, 1, False)
    _vanilla.load_state_dict(best_sd, strict=True)
    _vanilla.load_state_dict(last_sd, strict=True)
    del _vanilla

    mins = round((time.time() - t0) / 60.0, 2)
    tmp = ckpt_path.with_suffix(".tmp")
    torch.save({
        "best_model": best_sd,
        "last_model": last_sd,
        "best_dice": best_dice,
        "last_dice": last_dice,
        "aux_weight": aux_weight,
        "spine_gamma": spine_gamma,
        "history": history,
        "train_minutes": mins,
    }, tmp)
    os.replace(tmp, ckpt_path)
    done_path.write_text("ok", encoding="utf-8")
    del model, opt, scaler; torch.cuda.empty_cache()
    return ckpt_path, {"reused": False, "best_dice": best_dice, "last_dice": last_dice, "train_minutes": mins, "history": history}

ARM_CKPTS = {}
TRAIN_SUMMARY = {}
for _name, _cfg in CFG.ARMS.items():
    ARM_CKPTS[_name], TRAIN_SUMMARY[_name] = train_refiner_arm(_name, _cfg["aux_weight"], _cfg["spine_gamma"])
print("Refiner training summary:", {k: {kk: vv for kk, vv in v.items() if kk != "history"} for k, v in TRAIN_SUMMARY.items()})


In [ ]:
# [5/6] 厳密 OOF YOLO 提案の単一パス抽出・320x320 クロップキャッシュ構築
def mask_square_box(mask, context=CFG.CROP_CONTEXT, minimum=96):
    h, w = mask.shape
    ys, xs = np.where(mask)
    if not len(xs):
        return 0, 0, w, h
    cx, cy = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2
    side = int(math.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context))
    side = min(max(side, minimum), min(h, w))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > w: x0 -= x1 - w; x1 = w
    if y1 > h: y0 -= y1 - h; y1 = h
    return max(0, x0), max(0, y0), min(w, x1), min(h, y1)

cache_file = CFG.PROPOSAL_CACHE_DIR / f"oof_proposals_{CFG.SPLIT_MODE}_{len(valid_files)}.pkl"
YOLO_FOLD_SHAS = {}
eval_folds = sorted({fold_of_file[n] for n in valid_files})
for f in eval_folds:
    ypt = CFG.DRIVE_ROOT / "runs" / f"yolo_fold_{f}" / "weights" / "best.pt"
    assert ypt.exists(), f"Missing OOF YOLO checkpoint: {ypt}"
    YOLO_FOLD_SHAS[f"fold_{f}"] = file_sha256(ypt)

if cache_file.exists():
    EVAL_PROPOSALS = pickle.loads(cache_file.read_bytes())
    print(f"再利用: OOF 提案キャッシュ ({len(EVAL_PROPOSALS)} obs)")
else:
    by_fold = defaultdict(list)
    for n in valid_files:
        by_fold[fold_of_file[n]].append(n)
    EVAL_PROPOSALS = {}
    for f in eval_folds:
        ypt = CFG.DRIVE_ROOT / "runs" / f"yolo_fold_{f}" / "weights" / "best.pt"
        yolo_model = YOLO(str(ypt))
        for name in tqdm(by_fold[f], desc=f"OOF YOLO fold {f} 提案抽出"):
            img_path = CFG.PREPROC_DIR / name
            res = yolo_model.predict(
                str(img_path), imgsz=CFG.YOLO_IMGSZ, conf=CFG.PRIMARY_CONF,
                iou=CFG.PREDICT_IOU, agnostic_nms=True, retina_masks=True, verbose=False,
            )[0]
            h, w = res.orig_shape
            masks = []
            if res.masks is not None:
                for m in res.masks.data.detach().cpu().numpy():
                    if m.shape != (h, w):
                        m = cv2.resize(m, (w, h), interpolation=cv2.INTER_LINEAR)
                    masks.append(m >= 0.5)
            conf = res.boxes.conf.detach().cpu().numpy().astype(np.float32) if res.boxes is not None else np.zeros(0, np.float32)
            image = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
            items = []
            for i, pm in enumerate(masks):
                if pm.sum() < CFG.MIN_AREA:
                    continue
                x0, y0, x1, y1 = mask_square_box(pm)
                crop_u8 = cv2.resize(image[y0:y1, x0:x1], (CFG.CROP_SIZE, CFG.CROP_SIZE), interpolation=cv2.INTER_LINEAR)
                items.append({
                    "conf": float(conf[i]),
                    "raw_rle": mask_utils.encode(np.asfortranarray(pm.astype(np.uint8))),
                    "box": (x0, y0, x1, y1),
                    "crop_u8": crop_u8,
                })
            EVAL_PROPOSALS[name] = {"name": name, "fold": f, "shape": (h, w), "items": items}
        del yolo_model; torch.cuda.empty_cache()
    cache_file.write_bytes(pickle.dumps(EVAL_PROPOSALS, protocol=pickle.HIGHEST_PROTOCOL))
print("Total cached eval proposals:", sum(len(EVAL_PROPOSALS[n]["items"]) for n in valid_files))


In [ ]:
# [6/6] バッチ GPU Refiner 推論・厳密対照 PQ / Near-Miss / 対応あり Bootstrap 検証 & 保存
def drop_contained_indices(masks, threshold=CFG.DROP_THRESHOLD):
    if len(masks) < 2:
        return list(range(len(masks)))
    rles = mask_utils.encode(np.asfortranarray(np.stack(masks, axis=-1).astype(np.uint8)))
    areas = np.asarray(mask_utils.area(rles), dtype=np.float64)
    self_iou = np.asarray(mask_utils.iou(rles, rles, [0] * len(rles)), dtype=np.float64)
    dropped = set()
    for i in range(len(masks)):
        for j in range(i + 1, len(masks)):
            if i in dropped or j in dropped or self_iou[i, j] <= 0:
                continue
            inter = self_iou[i, j] * (areas[i] + areas[j]) / (1.0 + self_iou[i, j])
            smaller = i if areas[i] <= areas[j] else j
            if inter / max(areas[smaller], 1.0) >= threshold:
                dropped.add(smaller)
    return [i for i in range(len(masks)) if i not in dropped]

def apply_ownership_prod(masks):
    if not masks:
        return []
    areas = np.array([m.sum() for m in masks], dtype=np.float64)
    occupied = np.zeros(masks[0].shape, dtype=bool)
    kept = []
    for idx in sorted(drop_contained_indices(masks), key=lambda i: areas[i]):
        excl = masks[idx] & ~occupied
        if excl.sum() < CFG.MIN_AREA:
            continue
        kept.append(excl); occupied |= excl
    return kept

def iou_matrix(truth, pred):
    if not truth or not pred:
        return np.zeros((len(truth), len(pred)), np.float64)
    return np.asarray(mask_utils.iou(pred, truth, [0] * len(truth)), dtype=np.float64).T

def sparse_label_pq(sum_iou, tp, fp, fn, r=0.77):
    den = r * tp + 0.5 * fp + 0.5 * (1 - r) * tp + 0.5 * r * fn
    return float(r * sum_iou / den) if den > 0 else 0.0

def build_five_day_clusters(obs_names):
    parsed = sorted((datetime.strptime(n[:14], "%Y%m%d%H%M%S"), n) for n in obs_names)
    clusters, cur, prev_dt = [], [parsed[0][1]], parsed[0][0]
    for dt, name in parsed[1:]:
        if (dt - prev_dt).total_seconds() > 5 * 86400:
            clusters.append(cur); cur = [name]
        else:
            cur.append(name)
        prev_dt = dt
    clusters.append(cur)
    return clusters

def paired_bootstrap_pq(per_obs_a, per_obs_b, obs_names, block_mode="obs", n_boot=2000, seed=3407):
    rng = np.random.default_rng(seed)
    units = [[n] for n in obs_names] if block_mode == "obs" else build_five_day_clusters(obs_names)
    n_u = len(units)
    deltas = np.empty(n_boot, dtype=np.float64)
    for b in range(n_boot):
        pick = rng.integers(0, n_u, size=n_u)
        sa, sb = [0.0, 0, 0, 0], [0.0, 0, 0, 0]
        for ui in pick:
            for name in units[ui]:
                for k in range(4):
                    sa[k] += per_obs_a[name][k]
                    sb[k] += per_obs_b[name][k]
        den_a = sa[1] + 0.5 * sa[2] + 0.5 * sa[3]
        den_b = sb[1] + 0.5 * sb[2] + 0.5 * sb[3]
        deltas[b] = (sb[0] / den_b if den_b > 0 else 0.0) - (sa[0] / den_a if den_a > 0 else 0.0)
    return {
        "n_units": n_u,
        "block_mode": block_mode,
        "mean_delta": float(deltas.mean()),
        "ci95_low": float(np.quantile(deltas, 0.025)),
        "ci95_high": float(np.quantile(deltas, 0.975)),
        "prob_positive": float((deltas > 0).mean()),
    }

def predict_probs_for_refiner(ckpt_path: Path, state_key: str = "best_model"):
    net = MediSegEdgeAttUNet(3, 1, False).cuda().eval()
    ck = torch.load(ckpt_path, map_location="cuda", weights_only=False)
    net.load_state_dict(ck[state_key], strict=True)
    del ck
    probs_by_obs = {}
    with torch.no_grad():
        for name in valid_files:
            items = EVAL_PROPOSALS[name]["items"]
            if not items:
                probs_by_obs[name] = []
                continue
            arr = np.stack([it["crop_u8"].transpose(2, 0, 1) for it in items], axis=0)
            out_list = []
            for s in range(0, len(arr), 32):
                ten = torch.from_numpy(arr[s:s + 32]).float().cuda() / 255.0
                pr = net(ten).sigmoid()[:, 0].cpu().numpy()
                out_list.append(pr)
            probs_by_obs[name] = list(np.concatenate(out_list, axis=0))
    del net; torch.cuda.empty_cache()
    return probs_by_obs

def score_probs(probs_by_obs, thr: float):
    sum_iou, tp, fp, fn = 0.0, 0, 0, 0
    near_miss_fn = 0
    small_gt_total, small_hit = 0, 0
    per_obs = {}
    for name in valid_files:
        entry = EVAL_PROPOSALS[name]
        items = entry["items"]
        probs = probs_by_obs[name]
        refined_masks = []
        for it, prob_320 in zip(items, probs):
            pm = mask_utils.decode(it["raw_rle"]) != 0
            x0, y0, x1, y1 = it["box"]
            p_box = cv2.resize(prob_320, (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR)
            rm = np.zeros_like(pm, dtype=bool)
            rm[y0:y1, x0:x1] = p_box >= thr
            rm &= pm  # strict prod shrink_mask contract
            if rm.sum() >= CFG.MIN_AREA:
                refined_masks.append(rm)
        final_masks = apply_ownership_prod(refined_masks)
        pred_rles = mask_utils.encode(np.asfortranarray(np.stack(final_masks, -1).astype(np.uint8))) if final_masks else []
        obs_stat = [0.0, 0, 0, 0]
        for iid in file_to_ids[name]:
            truth = gt_rles[iid]
            areas = gt_areas[iid]
            mat = iou_matrix(truth, pred_rles)
            if len(truth) == 0:
                fp += len(pred_rles); obs_stat[2] += len(pred_rles)
            elif len(pred_rles) == 0:
                fn += len(truth); obs_stat[3] += len(truth)
            else:
                hits = mat > 0.5
                s_i = float(mat[hits].sum())
                t_i = int(hits.sum())
                fp_i = int((hits.sum(axis=0) == 0).sum())
                fn_i = int((hits.sum(axis=1) == 0).sum())
                sum_iou += s_i; tp += t_i; fp += fp_i; fn += fn_i
                obs_stat[0] += s_i; obs_stat[1] += t_i; obs_stat[2] += fp_i; obs_stat[3] += fn_i
                max_gt_iou = mat.max(axis=1)
                near_miss_fn += int(((max_gt_iou >= 0.30) & (max_gt_iou <= 0.50)).sum())
            sm_mask = areas <= CFG.SMALL_GT_MAX_AREA
            if sm_mask.any():
                sm_truth = [truth[k] for k in np.flatnonzero(sm_mask)]
                small_gt_total += len(sm_truth)
                if pred_rles:
                    pmat = iou_matrix(sm_truth, pred_rles)
                    small_hit += int((pmat.max(axis=1) > 0.5).sum())
        per_obs[name] = tuple(obs_stat)
    den = tp + 0.5 * fp + 0.5 * fn
    pq = sum_iou / den if den > 0 else 0.0
    sq = sum_iou / tp if tp else 0.0
    rq = tp / den if den > 0 else 0.0
    return {
        "pq": float(pq),
        "sq": float(sq),
        "rq": float(rq),
        "pq_sparse_r077": sparse_label_pq(sum_iou, tp, fp, fn, CFG.SPARSE_R),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "near_miss_fn_030_050": near_miss_fn,
        "small_gt_recall": small_hit / max(small_gt_total, 1),
    }, per_obs

bmode = "obs" if CFG.SPLIT_MODE == "pilot" else "5d_cluster"
PROBS_BEST = {arm: predict_probs_for_refiner(ARM_CKPTS[arm], "best_model") for arm in CFG.ARMS}
PROBS_LAST = {arm: predict_probs_for_refiner(ARM_CKPTS[arm], "last_model") for arm in CFG.ARMS}

PRIMARY_RESULTS = {}
PRIMARY_OBS = {}
THR_SWEEP = {}
LAST_MODEL_RESULTS = {}

for arm in CFG.ARMS:
    m_prim, obs_prim = score_probs(PROBS_BEST[arm], CFG.PRIMARY_THR)
    PRIMARY_RESULTS[arm] = m_prim
    PRIMARY_OBS[arm] = obs_prim
    m_last, _ = score_probs(PROBS_LAST[arm], CFG.PRIMARY_THR)
    LAST_MODEL_RESULTS[arm] = m_last
    THR_SWEEP[arm] = {f"thr_{t:.2f}": score_probs(PROBS_BEST[arm], t)[0] for t in CFG.DIAG_THRS}

COMPARISONS = {}
ctrl_pq = PRIMARY_RESULTS["ref_ctrl"]["pq"]
for arm in ("ref_spine_aux", "ref_spine_bal"):
    boot = paired_bootstrap_pq(PRIMARY_OBS["ref_ctrl"], PRIMARY_OBS[arm], valid_files, block_mode=bmode, n_boot=CFG.BOOTSTRAP_ITERS, seed=CFG.SEED)
    delta_pq = PRIMARY_RESULTS[arm]["pq"] - ctrl_pq
    delta_last = LAST_MODEL_RESULTS[arm]["pq"] - LAST_MODEL_RESULTS["ref_ctrl"]["pq"]
    COMPARISONS[f"{arm}_vs_ctrl"] = {
        "delta_pq_best_ckpt": delta_pq,
        "delta_pq_last_ckpt": delta_last,
        "delta_sq": PRIMARY_RESULTS[arm]["sq"] - PRIMARY_RESULTS["ref_ctrl"]["sq"],
        "delta_rq": PRIMARY_RESULTS[arm]["rq"] - PRIMARY_RESULTS["ref_ctrl"]["rq"],
        "delta_tp": PRIMARY_RESULTS[arm]["tp"] - PRIMARY_RESULTS["ref_ctrl"]["tp"],
        "delta_fp": PRIMARY_RESULTS[arm]["fp"] - PRIMARY_RESULTS["ref_ctrl"]["fp"],
        "delta_near_miss_fn": PRIMARY_RESULTS[arm]["near_miss_fn_030_050"] - PRIMARY_RESULTS["ref_ctrl"]["near_miss_fn_030_050"],
        "bootstrap": boot,
        "gate_pass": bool((delta_pq >= 0.005) and (boot["ci95_low"] > 0.0)),
    }

manifest_out = {
    "experiment_id": f"v129_spine_refiner_{CFG.SPLIT_MODE}",
    "runtime": RUNTIME_INFO,
    "hashes": {
        "annotations_sha256": file_sha256(TRAIN_JSON),
        "grouped_folds_sha256": folds_sha,
        "time_block_sha256": tb_sha,
        "exclusions_sha256": excl_sha,
        "yolo_oof_shas": YOLO_FOLD_SHAS,
        "refiner_arm_shas": {k: file_sha256(v) for k, v in ARM_CKPTS.items()},
    },
    "train_summary": TRAIN_SUMMARY,
    "primary_thr045_best_ckpt": PRIMARY_RESULTS,
    "last_epoch14_thr045": LAST_MODEL_RESULTS,
    "paired_comparisons_vs_ctrl": COMPARISONS,
    "threshold_sensitivity": THR_SWEEP,
}
manifest_path = CFG.ARTIFACT_ROOT / f"v129_spine_refiner_{CFG.SPLIT_MODE}_manifest.json"
manifest_path.write_text(json.dumps(manifest_out, indent=2, ensure_ascii=False), encoding="utf-8")

print("=== v129 SUMMARY TABLE (thr=0.45, strict shrink_mask inside OOF YOLO proposals) ===")
for arm in CFG.ARMS:
    r = PRIMARY_RESULTS[arm]
    rl = LAST_MODEL_RESULTS[arm]
    d = r["pq"] - ctrl_pq
    dl = rl["pq"] - LAST_MODEL_RESULTS["ref_ctrl"]["pq"]
    print(
        f"{arm:14s} | best_PQ={r['pq']:.5f} ({d:+.5f}) | last_PQ={rl['pq']:.5f} ({dl:+.5f}) | "
        f"SQ={r['sq']:.5f} RQ={r['rq']:.5f} | TP={r['tp']} FP={r['fp']} FN={r['fn']} near_miss={r['near_miss_fn_030_050']}"
    )
print("=== PAIRED BOOTSTRAP VS ref_ctrl ===")
print(json.dumps(COMPARISONS, indent=2, ensure_ascii=False))
print("Saved manifest:", manifest_path)
